In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch
import torchvision
from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from scipy.fftpack import fft, ifft
from sliced_wasserstein import sliced_wasserstein_distance
from c2st import c2st_knn, c2st_nn, c2st_rf

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

In [ ]:
#[1,2,13,24,25,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,91,92,102]
[46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,91,92,102]

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,91,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:
def load_model(cfg, start_index=100, subject_index=2):
    save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/model_checkpoints/finetune"
    #print(subject_index)
    file_path = os.path.join(save_path, f"subject_{subject_index}", f"model_checkpoint_finetune_subject_index_{subject_index}_start_idx_{start_index}_rep_0_pen.pth") 
    trunk_net = TrunkNet(n_chans=input_shape_st[0], n_times=input_shape_st[1])
    head_net = HeadNet(64, 1)  # Assuming these are the correct dimensions
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    
    weights = torch.load(file_path)
    model.load_state_dict(weights)
    #model.load_state_dict(full_checkpoint['model_state_dict'])
    model.eval()
    model.to(device)
    return model


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_shape_st = (60, 900)

In [ ]:
freq_bands = {"delta" : (0.5, 4),
            "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}
#amplification_factors = [0.2,0.5,2,3,5,10]
phase_peturbations = np.deg2rad(np.arange(45, 316, 45))

In [ ]:
cfg = load_config()

subject_index = cfg.dataset.test_subject_indices[1]
cfg.dataset.subject_index = subject_index
cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
cli_args = parse_args()
cfg = update_config(cfg, cli_args)
save_config(cfg)
all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg)
all_epochs = all_epochs[150:]
labels_raw = labels_raw[150:]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_shape_st = (60, 900)
del all_epochs
del labels_raw
del ch_names

In [ ]:
def amplify_frequency_band(signal, sampling_rate, low_freq, high_freq, amplification_factor):
    # FFT
    freqs = np.fft.rfftfreq(len(signal), d=1/sampling_rate)
    fft_coeffs = np.fft.rfft(signal)

    # Amplify the specified frequency band
    band_mask = (freqs >= low_freq) & (freqs <= high_freq)
    fft_coeffs[band_mask] *= amplification_factor

    # Inverse FFT
    modified_signal = np.fft.irfft(fft_coeffs, n=len(signal))
    return modified_signal

In [ ]:
def amplify_frequency_band_parallel(signals, sampling_rate, low_freq, high_freq, amplification_factor):
    # FFT
    freqs = np.fft.rfftfreq(signals.shape[1], d=1/sampling_rate)
    fft_coeffs = np.fft.rfft(signals, axis=1)

    # Amplify the specified frequency band
    band_mask = (freqs >= low_freq) & (freqs <= high_freq)
    fft_coeffs[:, band_mask] *= amplification_factor

    # Inverse FFT
    modified_signals = np.fft.irfft(fft_coeffs, n=signals.shape[1], axis=1)
    return modified_signals


In [ ]:
amplification_factors = [0.5, 0.75, 1.5,2 , 3 ]

In [ ]:
def create_perturbed_samples_dict(all_epochs, freq_bands, amplification_factors):
    perturbed_samples_dict = {}

    for band_name, (low_freq, high_freq) in freq_bands.items():
        for amp_factor in amplification_factors:
            #degree = np.rad2deg(phase_shift)
            perturbed_samples = np.zeros_like(all_epochs)

            for ch in range(all_epochs.shape[1]):
                perturbed_samples[:, ch] = amplify_frequency_band_parallel(all_epochs[:, ch], 1000, low_freq, high_freq, amp_factor)
            perturbed_samples_dict[(band_name, amp_factor)] = perturbed_samples

    return perturbed_samples_dict

# Example usage
#perturbed_samples_dict = create_perturbed_samples_dict(all_epochs, freq_bands, phase_peturbations)

In [ ]:
def get_predictions(dataset, start_index=100, subject_index=2):
    cfg = load_config()
    pred_label = np.zeros((dataset.shape[0]))
    uncertainties = np.zeros((dataset.shape[0]))
    input_shape_st = (60, 900)                
    for i in range(len(dataset)):
        current_start_index = i + start_index
        inputs = torch.from_numpy(dataset[i])
        inputs = inputs.to(device).float()
        inputs = inputs.unsqueeze(0)

        model = load_model(cfg, start_index=current_start_index, subject_index=subject_index)
        pred_mean, log_var = model(inputs)[:, 0], model(inputs)[:, 1]
        var = torch.exp(log_var)
        pred_label[i] = pred_mean.cpu().detach().numpy()
        uncertainties[i] = var.cpu().detach().numpy()
    
    return pred_label, uncertainties

In [ ]:

def compare_predictions(freq_bands, amplification_factors, predictions_perturbed, pred_label_original):
    #comparison_df = pd.DataFrame(index=amplification_factors, columns=freq_bands.keys())
    for band_name in freq_bands.keys():
        for factor in amplification_factors:
            key = f"{band_name}_{factor}"
            if key in predictions_perturbed:
                original_predictions = pred_label_original
                perturbed_predictions = predictions_perturbed[key]
                median_abs_diff = np.median(original_predictions - perturbed_predictions)
                #comparison_df.loc[factor, band_name] = mean_abs_diff
    #return comparison_df
    return median_abs_diff

In [ ]:
def pipeline_subject(subject_index, freq_bands, amplification_factors):
    # initialize config for subject
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    cli_args = parse_args()
    cfg = update_config(cfg, cli_args)
    save_config(cfg)

    # load data of subject
    all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    all_epochs = all_epochs[150:]
    labels_raw = labels_raw[150:]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    input_shape_st = (60, 900)
    # get predictions of original data
    original_predictions, original_uncertainties = get_predictions(all_epochs, subject_index=subject_index, start_index=100)
    # create perturbed samples
    perturbed_samples_dict = create_perturbed_samples_dict(all_epochs, freq_bands, amplification_factors)
    results_dict = {}
    perturbed_prediction_dict = {}
    
    for key, perturbed_samples in perturbed_samples_dict.items():
        # get predictions for perturbed samples
        pred_labels, uncertainties = get_predictions(perturbed_samples, subject_index=subject_index, start_index=100)
     
        #predictions_dict[key] = pred_labels
        #uncertainties_dict[key] = uncertainties
        result = np.median(original_predictions - pred_labels)
        results_dict[key] = result
    np.save(f"power_perturbation_results_{subject_index}.npy", results_dict)
    np.save(f"power_perturbed_predictions_{subject_index}.npy", perturbed_prediction_dict)

In [ ]:
#for subject_index in cfg.dataset.test_subject_indices:
#    pipeline_subject(subject_index, freq_bands, amplification_factors)

In [ ]:
def get_prediction_and_uncertainty(subject_index, freq_bands, amplification_factors):

    # initialize config for subject
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    cli_args = parse_args()
    cfg = update_config(cfg, cli_args)
    save_config(cfg)

    # load data of subject
    all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    all_epochs = all_epochs[150:]
    labels_raw = labels_raw[150:]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    input_shape_st = (60, 900)
    # get predictions of original data

    perturbed_samples_dict = create_perturbed_samples_dict(all_epochs, freq_bands, amplification_factors)
    preds = {}
    uncerts = {}
    for key, perturbed_samples in perturbed_samples_dict.items():
        # get predictions for perturbed samples
        pred_labels, uncertainties = get_predictions(perturbed_samples, subject_index=subject_index, start_index=100)
        preds[key] = pred_labels
        uncerts[key] = uncertainties
        
    dir = "all_power_perturb"
    os.makedirs(dir, exist_ok=True)
    save_path = os.path.join(dir, f"prediction_and_uncertanties_subject_{subject_index}_power")
    np.save(save_path, (preds, uncerts))


In [ ]:
for subject_index in cfg.dataset.test_subject_indices:
    get_prediction_and_uncertainty(subject_index, freq_bands, amplification_factors)